# ANKIT LLM v0.1 — **one-click** Colab T4 run (autopilot)

Tumhe kuch configure nahi karna. Bas **3 cells** chalao.

1. **Cell 1** — GPU check (T4 hona chahiye: *Runtime → Change runtime type → T4 GPU*)
2. **Cell 2** — Drive mount (ek Google login popup aayega, bas OK dabana hai)
3. **Cell 3** — autopilot: code clone, dependencies, data build, benchmark, **auto config**,
   pilot, resume test, aur (time bache to) long training + SFT + evaluation + chat demo.
   Sab kuch Drive me save hota hai.

Same cell 3 ko **agli session me dobara chalao** — training wahin se resume hogi.

Kuch manually edit nahi karna: batch size, dtype, configs — autopilot khud decide karta hai
(benchmark ke *measured* results se).


In [ ]:
!nvidia-smi
import torch, platform
print("py", platform.python_version(), "| torch", torch.__version__)
assert torch.cuda.is_available(), "GPU nahi mila: Runtime -> Change runtime type -> T4 GPU"
p = torch.cuda.get_device_properties(0); cap = torch.cuda.get_device_capability(0)
print("GPU:", p.name, "| cc", f"{cap[0]}.{cap[1]}", "| VRAM", round(p.total_memory/1e9,1), "GB")
print("bf16 accelerated:", cap[0] >= 8, "(T4 -> False, isliye fp16 + GradScaler use hoga)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = "/content/drive/MyDrive/ankit-llm"
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE}/reports", exist_ok=True)
print("drive ready:", DRIVE)

In [ ]:
# ⬇️ यही एक cell चलाना है (Run ▶). बाकी सब अपने आप होगा.
import os, subprocess, sys

REPO    = "https://github.com/pushparaj9749/tool.git"
REPO_DIR = "/content/ankit-llm"
BRANCHES = ["main", "arena/f3dbc078-tool"]   # script kis branch me mile, wahi use hoga

def run(cmd, **kw):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, **kw)

if not os.path.isdir(REPO_DIR):
    run(["git", "clone", REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# ⚠️ purana clone ho sakta hai (kal ka code) -> hamesha latest par laao.
# data/ aur checkpoints/ gitignored hain, isliye reset --hard unhe chhoota nahi.
run(["git", "fetch", "origin"], check=False, capture_output=True)
r = run(["git", "reset", "--hard", "origin/main"], check=False, capture_output=True, text=True)
print((r.stdout or r.stderr or "").strip()[-200:])

# agar script 'main' par nahi hai to (unmerged) feature branch par switch karo
if not os.path.exists("scripts/colab_autopilot.py"):
    for b in BRANCHES[1:]:
        r = run(["git", "fetch", "origin", b], capture_output=True, text=True)
        if r.returncode == 0:
            run(["git", "checkout", b], check=True)
            if os.path.exists("scripts/colab_autopilot.py"):
                break
assert os.path.exists("scripts/colab_autopilot.py"), "autopilot script repo me nahi mila"
print("branch:", run(["git","rev-parse","--abbrev-ref","HEAD"], capture_output=True, text=True).stdout.strip())

# Colab ka apna torch mat chhedo — sirf chhoti dependencies (pynvml = GPU utilisation)
run([sys.executable, "-m", "pip", "-q", "install",
     "pyyaml", "tokenizers", "numpy", "pynvml", "psutil", "pytest"], check=True)

# Colab limit se kam budget rakho (T4 ke liye 480 min = 8 ghante, safe)
SESSION_BUDGET_MIN = 480
BUILD = "t4"   # build-tag: isse pata chalta hai kaunsa code chal raha hai

# Output LIVE Drive me likha jayega — session crash ho jaye tab bhi log bacha rahega
import time
LOG_DIR = "/content/drive/MyDrive/ankit-llm/reports"
os.makedirs(LOG_DIR, exist_ok=True)
LOG = os.path.join(LOG_DIR, f"autopilot_{BUILD}_{time.strftime('%m%d_%H%M')}.log")
cmd = [sys.executable, "-u", "scripts/colab_autopilot.py",
       "--mode", "all", "--session-budget-min", str(SESSION_BUDGET_MIN)]
env = dict(os.environ, PYTHONUNBUFFERED="1")
print("log ->", LOG, "\n", flush=True)
with open(LOG, "w", encoding="utf-8") as fh:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
        fh.write(line); fh.flush()
rc = proc.wait()
print(f"\nreturncode = {rc}  |  log saved: {LOG}")


## Result dekho

`reports/autopilot_summary.md` (Drive me bhi copy hota hai) me sab measured numbers honge:
benchmark table, auto-selected config, pilot verification, checkpoints.

**Yahi file ka content mujhe bhej dena** — usme sab kuch hoga.

## Chat try karo (GPU par fast hai)

```python
from inference.conversation import ChatEngine
eng = ChatEngine.from_checkpoint("checkpoints/t4-sft/final.pt", device="cuda")
print(eng.chat_once([{"role":"user","content":"नमस्ते अंकित"}], mode="adaptive"))
```

## Agli session (training continue)

Wahi cell 3 dobara chalao. `--resume auto` naya checkpoint Drive se utha kar
usi LR schedule par continue karega.
